# 01 — Cognitive Substrate

**An agent that knows when it's struggling.**

Orqest's cognitive substrate is built on one handshake: *metacognition produces the signal; healing acts on it.* This notebook wires that edge end to end —

1. an agent self-assesses its own output (`run_enriched` + a `ConfidenceProtocol`),
2. a `MetacognitionHook` publishes that confidence onto an `EventBus`,
3. a `RegressionDetector` watches the confidence trend and raises a `Detection`,
4. a `WatchdogHook` turns the detection into a `HookDecision` that halts the run.

**Prerequisites:** a `.env` at the repo root with `LLM_API_KEY` and `LLM_MODEL` (see the main README).

## 1. Load config

In [1]:
from orqest import load_config

config = load_config()
print(f"Model: {config.llm_model}")

Model: openai:gpt-5.2


## 2. An agent that self-assesses

The zero-cost `StructuredOutputProtocol` lifts three fields straight off the agent's own output model — `self_confidence`, `uncertain_about`, `outside_my_capability`. The agent just has to *declare* them; the LLM fills them in as part of its structured response.

We pass the protocol as a constructor-level default, so `run_enriched` uses it automatically. The constructor takes `model` as either a `provider:model_id` string or a pre-built `Model` — we'll use the latter in section 6.

In [2]:
from pydantic import BaseModel, Field
from orqest.agents import BaseAgent, GlobalState
from orqest.metacognition import StructuredOutputProtocol


class AnswerOutput(BaseModel):
    """Structured answer + the agent's honest self-assessment."""
    answer: str = Field(description="The answer to the user's question.")
    self_confidence: float | None = Field(
        default=None,
        description="Your honest probability (0-1) that this answer is correct.",
    )
    uncertain_about: list[str] = Field(
        default_factory=list,
        description="Specific sub-claims or assumptions you are unsure about.",
    )
    outside_my_capability: bool = Field(
        default=False,
        description="True if the question needs information you cannot have "
        "(future events, private data, post-cutoff knowledge).",
    )


class ResearchAgent(BaseAgent[GlobalState, AnswerOutput]):
    def __init__(self, model, api_key: str | None = None):
        super().__init__(
            agent_name="research_agent",
            system_prompt=(
                "You answer questions concisely. Be ruthlessly honest in your "
                "self-assessment: set self_confidence LOW when you are guessing "
                "at a specific fact, list what you are unsure about, and set "
                "outside_my_capability=True when a question needs information "
                "no one could have yet."
            ),
            output_type=AnswerOutput,
            model=model,
            api_key=api_key,
            confidence_protocol=StructuredOutputProtocol(),
        )

    async def _run_implementation(self, state: GlobalState, **kwargs) -> AnswerOutput:
        result = await self.call_model(state.get_latest_message("user"), state)
        return result.output


agent = ResearchAgent(model=config.llm_model, api_key=config.llm_api_key)

## 3. `run_enriched` — output paired with self-assessment

`run` returns the bare `AnswerOutput`. `run_enriched` returns an `EnrichedOutput[AnswerOutput]` — the same output, plus a normalised `confidence`, `uncertainty_targets`, and a `capability_boundary` flag. The underlying `run` is untouched; enrichment is purely additive.

In [3]:
async def ask(question: str):
    state = GlobalState()
    state.add_message("user", question)
    enriched = await agent.run_enriched(state)
    print(f"Q: {question}")
    print(f"   answer:     {enriched.output.answer[:90]}")
    print(f"   confidence: {enriched.confidence}")
    print(f"   uncertain:  {enriched.uncertainty_targets}")
    print(f"   boundary:   {enriched.capability_boundary}")
    return enriched


_ = await ask("What is the capital of France?")

Q: What is the capital of France?
   answer:     Paris.
   confidence: 0.99
   uncertain:  []
   boundary:   False


In [4]:
_ = await ask("What will the GBP/USD exchange rate be next Tuesday?")

Q: What will the GBP/USD exchange rate be next Tuesday?
   answer:     I can’t reliably predict the exact GBP/USD exchange rate for next Tuesday. Exchange rates 
   confidence: 0.95
   uncertain:  []
   boundary:   True


Two things to notice — and the second is a real lesson about confidence:

- The unanswerable question flips `capability_boundary` to **True** and fills `uncertainty_targets`. That flag is a clean, reliable signal.
- The headline `confidence` may *still be high* on that same question — the model is (correctly) confident that *"I can't know this"* is the right response. **`self_confidence` measures confidence in the response, not the difficulty of the question.** Keep that in mind when you wire it into healing.

## 4. The handshake — confidence onto the bus

`MetacognitionHook` is a `ToolHook`: whenever a compound-tool call returns an `EnrichedOutput`, it publishes a `metacognition.confidence` event on the `EventBus`. `RegressionDetector` subscribes to exactly that event type and buffers a sliding window of confidence values.

Here we run the agent over a few prompts and feed each enrichment through the hook — the same `after_tool` call the hook makes automatically inside a `CompoundTool` / `SubAgentTool` flow. The point of this section is the *plumbing*: agent → hook → bus → detector. (As section 3 showed, real-LLM `confidence` is noisy — the detector consumes whatever values flow; section 5 drives it deterministically.)

In [5]:
from orqest.observability import EventBus
from orqest.metacognition import MetacognitionHook
from orqest.healing import RegressionDetector

bus = EventBus()
detector = RegressionDetector(window_n=4, drop_threshold=0.15)
detector.subscribe(bus)

metacog_hook = MetacognitionHook(bus, agent_name="research_agent")

prompts = [
    "What is the boiling point of water at sea level, in Celsius?",
    "Summarise the main causes of the 2008 financial crisis in two sentences.",
    "Name the single most valuable startup that will be founded next year.",
]

for p in prompts:
    state = GlobalState()
    state.add_message("user", p)
    enriched = await agent.run_enriched(state)
    # Exactly what MetacognitionHook does after a compound-tool call:
    await metacog_hook.after_tool("research", {}, enriched, state, 0.0)
    print(f"confidence={enriched.confidence}  boundary={enriched.capability_boundary}"
          f"  ::  {p[:50]}")

confidence=0.99  boundary=False  ::  What is the boiling point of water at sea level, i


confidence=0.93  boundary=False  ::  Summarise the main causes of the 2008 financial cr


confidence=0.95  boundary=True  ::  Name the single most valuable startup that will be


## 5. Detection → decision

To see `RegressionDetector` fire deterministically, we feed it a controlled declining-confidence sequence. In production these values arrive live from `MetacognitionHook` (section 4) — here we emit them directly so the mechanism is visible.

`WatchdogHook` polls each watchdog's `signal()`, runs the policy (`default_policy` maps every detection to `AbortRun`), and returns a `HookDecision`. Dropped through a `HookRunner` — the same aggregation every other hook flows through — an `Abort` surfaces as a `HookAbortError`, which the surrounding `CompoundTool` / `MetaOrchestrator` / `run_with_retry` treats as a halt.

In [6]:
from orqest.observability import AgentEvent
from orqest.healing import WatchdogHook, default_policy
from orqest.hooks import HookRunner, HookAbortError

# A controlled declining-confidence sequence.
demo_bus = EventBus()
demo_detector = RegressionDetector(window_n=4, drop_threshold=0.15)
demo_detector.subscribe(demo_bus)
for conf in [0.97, 0.92, 0.45, 0.20]:
    await demo_bus.emit(AgentEvent(
        event_type="metacognition.confidence",
        agent_name="research_agent",
        data={"confidence": conf},
    ))

runner = HookRunner([WatchdogHook([demo_detector], policy=default_policy)])
try:
    await runner.run_before("next_research_step", {}, None)
    print("no halt — confidence trend is stable")
except HookAbortError as exc:
    print(f"HookAbortError raised — {exc}")

HookAbortError raised — regression: Confidence dropped from 0.95 → 0.33


## 6. Model fallback (the other healing axis)

`RegressionDetector` watches *the agent*. `FallbackModel` watches *the provider*: it wraps a chain of models and, on a transient failure (5xx, timeout, rate-limit), sticks to the next one. It is a `pydantic_ai.Model` subclass, so it drops straight into any `BaseAgent(model=...)`.

`resolve_model_with_fallback` builds one from `provider:model_id` strings. A real chain lists two or more models; here we resolve a one-element chain just to show the wiring — and because our `ResearchAgent` accepts a `Model` instance, it slots straight in.

In [7]:
from orqest.healing import resolve_model_with_fallback

fallback_model = resolve_model_with_fallback(
    [config.llm_model],
    api_key=config.llm_api_key,
)
print(f"fallback model: {fallback_model.model_name}")

# Slots directly into the agent — the agent loop is unchanged.
resilient_agent = ResearchAgent(model=fallback_model)
print(f"agent model:    {resilient_agent.model.model_name}")

fallback model: fallback(gpt-5.2)
agent model:    fallback(gpt-5.2)


## What's happening under the hood

- `run_enriched` runs `_run_implementation` untouched, then hands the output to the `ConfidenceProtocol`. `StructuredOutputProtocol` reads the three declared fields — zero extra LLM calls.
- `MetacognitionHook.after_tool` only fires when the tool result *is* an `EnrichedOutput`; on anything else it's a no-op, so it's safe to register on every `HookRunner`.
- `RegressionDetector` fires when the head-half mean of its confidence window exceeds the tail-half mean by `drop_threshold` — robust against single-sample noise without full statistics.
- `default_policy` is deliberately conservative: every detection becomes `AbortRun`. Override it with a custom callable once you've measured a recovery action that's safe in your domain.
- **Calibration is on you.** As section 3 showed, raw LLM `self_confidence` is noisy. `capability_boundary` is the more reliable signal; for trend detection, consider an `EnsembleProtocol` (agreement-based) or your own `ConfidenceProtocol` rather than trusting a single self-rating.
- In production you don't poll watchdogs by hand — a `HealingRunner` (or `Workbench.with_healing(config)`) owns the bus subscriptions and a poll loop. This notebook unrolls that loop so each step is visible.